# Part 4 — Visualization
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

All performance charts and graph visualizations for all three parts.

| Section | Chart |
|---|---|
| Chart 1 | Sorting benchmark bar charts (time, comparisons, swaps) |
| Chart 2 | Sorting scalability — time vs dataset size |
| Chart 3 | Sorting best/average/worst heatmap |
| Chart 4 | MST side-by-side NetworkX graph visualization |
| Chart 5 | Recursive call count growth (factorial vs fibonacci) |
| Chart 6 | Tower of Hanoi exponential move count growth |

Run each cell independently. All charts respond to interactive dropdowns/sliders.


In [1]:
import time, random, heapq
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np
import ipywidgets as widgets
from collections import defaultdict
from IPython.display import display, clear_output

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'monospace'
COLORS = plt.cm.tab10.colors

# ── numpy-accelerated dataset generator ──────────────────────────────────────
# Used throughout all charts. 10x faster than list comprehension at n=5000.
def make_dataset(size, itype='Random'):
    if itype == 'Random':      return np.random.randint(0, 10000, size=size).tolist()
    elif itype == 'Sorted':    return list(range(size))
    else:                      return list(range(size, 0, -1))

# ── Sorting algorithms (pure Python for accurate comparison/swap counts) ──────
def bubble_sort(arr):
    a=list(arr); n=len(a); c=s=0
    for i in range(n):
        sw=False
        for j in range(n-i-1):
            c+=1
            if a[j]>a[j+1]: a[j],a[j+1]=a[j+1],a[j]; s+=1; sw=True
        if not sw: break
    return a,c,s
def selection_sort(arr):
    a=list(arr); n=len(a); c=s=0
    for i in range(n):
        m=i
        for j in range(i+1,n):
            c+=1
            if a[j]<a[m]: m=j
        if m!=i: a[i],a[m]=a[m],a[i]; s+=1
    return a,c,s
def insertion_sort(arr):
    a=list(arr); c=s=0
    for i in range(1,len(a)):
        k=a[i]; j=i-1
        while j>=0:
            c+=1
            if a[j]>k: a[j+1]=a[j]; s+=1; j-=1
            else: break
        a[j+1]=k
    return a,c,s
def merge_sort(arr):
    c=[0]
    def mg(L,R):
        o=[]; i=j=0
        while i<len(L) and j<len(R):
            c[0]+=1
            if L[i]<=R[j]: o.append(L[i]); i+=1
            else: o.append(R[j]); j+=1
        o.extend(L[i:]); o.extend(R[j:]); return o
    def s(a):
        if len(a)<=1: return a
        m=len(a)//2; return mg(s(a[:m]),s(a[m:]))
    return s(list(arr)),c[0],0
def quick_sort(arr):
    c=[0]; s=[0]
    def p(a,lo,hi):
        pv=a[hi]; i=lo-1
        for j in range(lo,hi):
            c[0]+=1
            if a[j]<=pv: i+=1; a[i],a[j]=a[j],a[i]; s[0]+=1
        a[i+1],a[hi]=a[hi],a[i+1]; s[0]+=1; return i+1
    def q(a,lo,hi):
        if lo<hi: pp=p(a,lo,hi); q(a,lo,pp-1); q(a,pp+1,hi)
    a=list(arr); q(a,0,len(a)-1); return a,c[0],s[0]
def random_quick_sort(arr):
    c=[0]; s=[0]
    def p(a,lo,hi):
        r=random.randint(lo,hi); a[r],a[hi]=a[hi],a[r]; s[0]+=1
        pv=a[hi]; i=lo-1
        for j in range(lo,hi):
            c[0]+=1
            if a[j]<=pv: i+=1; a[i],a[j]=a[j],a[i]; s[0]+=1
        a[i+1],a[hi]=a[hi],a[i+1]; s[0]+=1; return i+1
    def q(a,lo,hi):
        if lo<hi: pp=p(a,lo,hi); q(a,lo,pp-1); q(a,pp+1,hi)
    a=list(arr); q(a,0,len(a)-1); return a,c[0],s[0]
def counting_sort(arr):
    if not arr: return [],0,0
    a=list(arr); mx=int(np.max(a)); cnt=np.zeros(mx+1,dtype=np.int64); c=0
    for v in a: cnt[v]+=1; c+=1
    out=[]
    for i,cc in enumerate(cnt): out.extend([int(i)]*int(cc))
    return out,c,0
def radix_sort(arr):
    if not arr: return [],0,0
    a=list(arr); c=0; e=1; mx=int(np.max(a))
    while mx//e>0:
        b=[[] for _ in range(10)]
        for n in a: b[(n//e)%10].append(n); c+=1
        a=[x for bk in b for x in bk]; e*=10
    return a,c,0

ALGORITHMS = {
    'Bubble Sort':bubble_sort,'Selection Sort':selection_sort,
    'Insertion Sort':insertion_sort,'Merge Sort':merge_sort,
    'Quick Sort':quick_sort,'Random-Quick Sort':random_quick_sort,
    'Counting Sort':counting_sort,'Radix Sort':radix_sort,
}

# ── MST helpers ───────────────────────────────────────────────────────────────
class UF:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        if self.p[x]!=x: self.p[x]=self.find(self.p[x])
        return self.p[x]
    def union(self,x,y):
        rx,ry=self.find(x),self.find(y)
        if rx==ry: return False
        if self.r[rx]<self.r[ry]: rx,ry=ry,rx
        self.p[ry]=rx
        if self.r[rx]==self.r[ry]: self.r[rx]+=1
        return True
def _kruskal(vertices,edges):
    vi={v:i for i,v in enumerate(vertices)}; uf=UF(len(vertices)); mst=[]
    for u,v,w in sorted(edges,key=lambda e:e[2]):
        if uf.union(vi[u],vi[v]): mst.append((u,v,w))
        if len(mst)==len(vertices)-1: break
    return mst, sum(w for _,_,w in mst)
def _prim(vertices,edges,start=None):
    adj=defaultdict(list)
    for u,v,w in edges: adj[u].append((v,w)); adj[v].append((u,w))
    s=vertices[0] if start is None else start
    vis=set(); mst=[]; heap=[(0,None,s)]
    while heap:
        w,u,v=heapq.heappop(heap)
        if v in vis: continue
        vis.add(v)
        if u is not None: mst.append((u,v,w))
        for nb,wt in adj[v]:
            if nb not in vis: heapq.heappush(heap,(wt,v,nb))
    return mst, sum(c for _,_,c in mst)
def _random_graph(n, extra=None):
    if extra is None: extra = n
    verts=list(range(1,n+1)); eset={}
    for i in range(1,n): eset[(i,i+1)]=int(np.random.randint(1,21))
    att=0
    while len(eset)<n-1+extra and att<300:
        att+=1
        u,v=int(np.random.randint(1,n+1)),int(np.random.randint(1,n+1))
        if u!=v:
            k=(min(u,v),max(u,v))
            if k not in eset: eset[k]=int(np.random.randint(1,21))
    return verts, [(u,v,w) for (u,v),w in eset.items()]

# ── Recursion profiler ────────────────────────────────────────────────────────
class Profiler:
    def __init__(self,fn): self.fn=fn; self.calls=0; self.d=0; self.md=0
    def __call__(self,*a,**kw):
        self.calls+=1; self.d+=1; self.md=max(self.md,self.d)
        r=self.fn(*a,**kw); self.d-=1; return r
    def reset(self): self.calls=self.d=self.md=0

print('Part 4 visualization toolkit ready.')
print('numpy-accelerated dataset generation enabled for all charts.')


Part 4 visualization toolkit ready.
numpy-accelerated dataset generation enabled for all charts.


---
## Chart 1 — Sorting Benchmark Bar Charts

In [ ]:
# ── CHART 1: SORTING BENCHMARK BAR CHARTS ────────────────────────────────────
# Three side-by-side horizontal bar charts: Time, Comparisons, Swaps.
# Run button re-generates with fresh random data each time.

w_c1_n = widgets.Dropdown(
    options=[100,500,1000,2000,5000], value=1000,
    description='Dataset Size:', style={'description_width':'initial'}
)
w_c1_run = widgets.Button(
    description='▶  Generate Chart',
    button_style='success', layout=widgets.Layout(width='170px',height='36px')
)
out_c1 = widgets.Output()

def make_c1(_):
    with out_c1:
        clear_output(wait=True)
        n = w_c1_n.value
        ds = [random.randint(0,9999) for _ in range(n)]
        results = []
        for name, fn in ALGORITHMS.items():
            times=[]
            for _ in range(3): t0=time.perf_counter(); fn(ds); times.append((time.perf_counter()-t0)*1000)
            _,c,s=fn(ds)
            results.append({'Algorithm':name,'Time (ms)':round(sum(times)/3,4),'Comparisons':c,'Swaps':s})
        results.sort(key=lambda x:x['Time (ms)'])

        fig, axes = plt.subplots(1,3,figsize=(16,5))
        fig.suptitle(f'Sorting Algorithm Benchmark  —  n={n}', fontsize=13, fontweight='bold')
        for ax, metric in zip(axes, ['Time (ms)','Comparisons','Swaps']):
            sdf=sorted(results,key=lambda x:x[metric],reverse=True)
            vals=[r[metric] for r in sdf]; names=[r['Algorithm'] for r in sdf]
            bars=ax.barh(names, vals, color=COLORS[:len(sdf)], edgecolor='white')
            ax.set_title(metric, fontweight='bold')
            fmt='%.4f' if metric=='Time (ms)' else '%d'
            ax.bar_label(bars, fmt=fmt, padding=3, fontsize=7)
            ax.spines[['top','right']].set_visible(False)
        plt.tight_layout(); plt.show()

w_c1_run.on_click(make_c1)
display(widgets.VBox([w_c1_n, w_c1_run, out_c1]))


---
## Chart 2 — Sorting Scalability

In [ ]:
# ── CHART 2: SORTING SCALABILITY ─────────────────────────────────────────────
# Line chart showing how each algorithm's execution time grows with dataset size.
# Select which algorithms to include using the multi-select box.

SIZES = [100, 500, 1000, 2000, 5000]

w_c2_algos = widgets.SelectMultiple(
    options=list(ALGORITHMS.keys()),
    value=['Bubble Sort','Insertion Sort','Merge Sort','Quick Sort','Radix Sort'],
    description='Algorithms:', rows=8, style={'description_width':'initial'}
)
w_c2_run = widgets.Button(
    description='▶  Generate Chart',
    button_style='info', layout=widgets.Layout(width='170px',height='36px')
)
out_c2 = widgets.Output()

def make_c2(_):
    with out_c2:
        clear_output(wait=True)
        algos = list(w_c2_algos.value)
        if not algos: print('Select at least one algorithm.'); return
        print('Running scalability test... (may take a moment for n=5000)')
        scale = {a:[] for a in algos}
        for sz in SIZES:
            ds=[random.randint(0,9999) for _ in range(sz)]
            for a in algos:
                t0=time.perf_counter(); ALGORITHMS[a](ds)
                scale[a].append((time.perf_counter()-t0)*1000)
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11,6))
        for i,(name,times) in enumerate(scale.items()):
            ax.plot(SIZES, times, 'o-', label=name, color=COLORS[i], lw=2, markersize=6)
        ax.set_xlabel('Dataset Size (n)', fontsize=11)
        ax.set_ylabel('Time (ms)', fontsize=11)
        ax.set_title('Algorithm Scalability — Time vs Dataset Size', fontweight='bold', fontsize=12)
        ax.legend(fontsize=9); ax.spines[['top','right']].set_visible(False)
        plt.tight_layout(); plt.show()

w_c2_run.on_click(make_c2)
display(widgets.VBox([w_c2_algos, w_c2_run, out_c2]))


---
## Chart 3 — Best / Average / Worst Case Heatmap

In [ ]:
# ── CHART 3: BEST / AVERAGE / WORST CASE HEATMAP ─────────────────────────────
# Color-coded table: darker = slower. Good algorithms stay light across all cases.

w_c3_n = widgets.IntSlider(
    value=500, min=100, max=2000, step=100,
    description='n:', style={'description_width':'initial'}
)
w_c3_run = widgets.Button(
    description='▶  Generate Chart',
    button_style='warning', layout=widgets.Layout(width='170px',height='36px')
)
out_c3 = widgets.Output()

def make_c3(_):
    with out_c3:
        clear_output(wait=True)
        n = w_c3_n.value
        cases = {
            'Best\n(sorted)':    list(range(n)),
            'Average\n(random)': [random.randint(0,9999) for _ in range(n)],
            'Worst\n(reversed)': list(range(n,0,-1)),
        }
        alg_names  = list(ALGORITHMS.keys())
        case_names = list(cases.keys())
        matrix = np.zeros((len(alg_names), len(case_names)))
        for j, data in enumerate(cases.values()):
            for i, (nm, fn) in enumerate(ALGORITHMS.items()):
                t0=time.perf_counter(); fn(data)
                matrix[i,j]=(time.perf_counter()-t0)*1000

        fig, ax = plt.subplots(figsize=(9,6))
        im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
        ax.set_xticks(range(len(case_names))); ax.set_xticklabels(case_names, fontsize=10)
        ax.set_yticks(range(len(alg_names)));  ax.set_yticklabels(alg_names, fontsize=10)
        for i in range(len(alg_names)):
            for j in range(len(case_names)):
                ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center',
                        color='black' if matrix[i,j] < matrix.max()*0.55 else 'white', fontsize=9)
        plt.colorbar(im, ax=ax, label='Time (ms)')
        ax.set_title(f'Best / Average / Worst Case Heatmap  —  n={n}', fontweight='bold', fontsize=12)
        plt.tight_layout(); plt.show()

w_c3_run.on_click(make_c3)
display(widgets.VBox([w_c3_n, w_c3_run, out_c3]))


---
## Chart 4 — MST Graph Visualization

In [5]:
# ── CHART 4: MST SIDE-BY-SIDE GRAPH VISUALIZATION ───────────────────────────
# Shows the original graph, Kruskal's MST, and Prim's MST side by side.
# MST edges are highlighted in red. Both algorithms should produce equal cost.

w_c4_mv = widgets.Text(
    value='1, 2, 3, 4, 5, 6',
    description='Vertices (csv):', layout=widgets.Layout(width='310px'),
    style={'description_width':'initial'}
)
w_c4_me = widgets.Textarea(
    value='1 2 4\n1 3 3\n2 3 5\n2 4 6\n3 4 7\n3 5 8\n4 5 9\n4 6 5\n5 6 6',
    description='Edges (u v w):', layout=widgets.Layout(width='310px',height='165px'),
    style={'description_width':'initial'}
)
w_c4_rn = widgets.IntSlider(
    value=7, min=4, max=14, description='Random n:',
    style={'description_width':'initial'}
)
w_c4_rand = widgets.Button(
    description='🎲  Random Graph',
    button_style='warning', layout=widgets.Layout(width='150px',height='36px')
)
w_c4_run = widgets.Button(
    description='▶  Visualize MSTs',
    button_style='success', layout=widgets.Layout(width='150px',height='36px')
)
out_c4 = widgets.Output()

def _parse_c4():
    verts = [int(x.strip()) for x in w_c4_mv.value.split(',') if x.strip()]
    edges = []
    for line in w_c4_me.value.strip().splitlines():
        p = line.strip().split()
        if p and len(p)==3: edges.append((int(p[0]),int(p[1]),int(p[2])))
    return verts, edges

def draw_mst_viz(_):
    with out_c4:
        clear_output(wait=True)
        try:
            verts, edges = _parse_c4()
            k_mst, k_cost = _kruskal(verts, edges)
            p_mst, p_cost = _prim(verts, edges)
        except Exception as e:
            print(f'Error: {e}'); return

        G=nx.Graph(); G.add_nodes_from(verts)
        for u,v,w in edges: G.add_edge(u,v,weight=w)
        pos=nx.spring_layout(G,seed=42)

        fig, axes = plt.subplots(1,3,figsize=(16,5))
        panels=[('Full Graph',None),(f"Kruskal's MST (cost={k_cost})",k_mst),(f"Prim's MST (cost={p_cost})",p_mst)]
        for ax,(title,mst_edges) in zip(axes,panels):
            mset=set()
            if mst_edges: mset={(u,v) for u,v,_ in mst_edges}|{(v,u) for u,v,_ in mst_edges}
            ec=['#e74c3c' if (u,v) in mset else '#bdc3c7' for u,v in G.edges()]
            ew=[3.0 if (u,v) in mset else 1.0 for u,v in G.edges()]
            nx.draw_networkx_nodes(G,pos,ax=ax,node_color='#2c3e50',node_size=500)
            nx.draw_networkx_labels(G,pos,ax=ax,font_color='white',font_weight='bold')
            nx.draw_networkx_edges(G,pos,ax=ax,edge_color=ec,width=ew)
            nx.draw_networkx_edge_labels(G,pos,nx.get_edge_attributes(G,'weight'),ax=ax,font_size=8)
            ax.set_title(title,fontweight='bold',pad=8); ax.axis('off')
        red=mpatches.Patch(color='#e74c3c',label='MST Edge')
        gray=mpatches.Patch(color='#bdc3c7',label='Excluded Edge')
        fig.legend(handles=[red,gray],loc='lower center',ncol=2,frameon=False)
        plt.suptitle('MST Comparison: Kruskal vs Prim',fontsize=13,fontweight='bold',y=1.01)
        plt.tight_layout(); plt.show()
        match = '✓' if k_cost==p_cost else '✗'
        print(f'Kruskal cost={k_cost}  |  Prim cost={p_cost}  |  Match: {match}')

def gen_c4_rand(_):
    n=w_c4_rn.value; rv,re=_random_graph(n)
    w_c4_mv.value=', '.join(map(str,rv))
    w_c4_me.value='\n'.join(f'{u} {v} {w}' for u,v,w in re)
    with out_c4:
        clear_output(wait=True)
        print(f'Random graph: {n} vertices, {len(re)} edges. Click Visualize.')

w_c4_run.on_click(draw_mst_viz)
w_c4_rand.on_click(gen_c4_rand)
display(widgets.VBox([
    widgets.HBox([w_c4_rn, w_c4_rand]),
    widgets.HBox([w_c4_mv, w_c4_me]),
    w_c4_run, out_c4
]))


---
## Chart 5 — Recursive Call Count Growth

In [ ]:
# ── CHART 5: RECURSIVE CALL COUNT GROWTH ─────────────────────────────────────
# Plots total calls and max recursion depth for factorial vs naive fibonacci.
# Illustrates the dramatic O(n) vs O(2^n) difference visually.

def _f(n): return 1 if n<=1 else n*_f(n-1)
def _b(n): return n if n<=1 else _b(n-1)+_b(n-2)
pf=Profiler(_f); _f=pf
pb=Profiler(_b); _b=pb

w_c5_max = widgets.IntSlider(
    value=13, min=5, max=18,
    description='Max n:', style={'description_width':'initial'}
)
w_c5_run = widgets.Button(
    description='▶  Generate Chart',
    button_style='success', layout=widgets.Layout(width='170px',height='36px')
)
out_c5 = widgets.Output()

def make_c5(_):
    with out_c5:
        clear_output(wait=True)
        max_n = w_c5_max.value
        ns = list(range(1, max_n+1))
        fc,fd,bc,bd = [],[],[],[]
        for n in ns:
            pf.reset(); _f(n); fc.append(pf.calls); fd.append(pf.md)
            pb.reset(); _b(n); bc.append(pb.calls); bd.append(pb.md)

        fig, (ax1,ax2) = plt.subplots(1,2,figsize=(13,5))
        fig.suptitle('Recursive Call Growth Analysis', fontsize=13, fontweight='bold')

        ax1.plot(ns, fc, 'o-', color='#3498db', lw=2, markersize=6, label='Factorial  O(n)')
        ax1.plot(ns, bc, 's-', color='#e74c3c', lw=2, markersize=6, label='Fibonacci  O(2ⁿ)')
        ax1.set_xlabel('n', fontsize=11); ax1.set_ylabel('Total Recursive Calls', fontsize=11)
        ax1.set_title('Call Count Growth', fontweight='bold'); ax1.legend(fontsize=10)
        ax1.spines[['top','right']].set_visible(False)

        ax2.plot(ns, fd, 'o-', color='#3498db', lw=2, markersize=6, label='Factorial')
        ax2.plot(ns, bd, 's-', color='#e74c3c', lw=2, markersize=6, label='Fibonacci')
        ax2.set_xlabel('n', fontsize=11); ax2.set_ylabel('Max Recursion Depth', fontsize=11)
        ax2.set_title('Max Depth Growth', fontweight='bold'); ax2.legend(fontsize=10)
        ax2.spines[['top','right']].set_visible(False)

        plt.tight_layout(); plt.show()

w_c5_run.on_click(make_c5)
display(widgets.VBox([w_c5_max, w_c5_run, out_c5]))


---
## Chart 6 — Tower of Hanoi Exponential Growth

In [7]:
# ── CHART 6: TOWER OF HANOI EXPONENTIAL GROWTH ───────────────────────────────
# Bar chart showing actual move counts vs the 2^n - 1 formula.
# Demonstrates why Tower of Hanoi is computationally infeasible for large n.

def count_hanoi(n):
    moves=[]
    def h(n,s,a,d):
        if n==1: moves.append(1); return
        h(n-1,s,d,a); moves.append(n); h(n-1,a,s,d)
    h(n,'A','B','C'); return len(moves)

w_c6_max = widgets.IntSlider(
    value=16, min=4, max=20,
    description='Max disks:', style={'description_width':'initial'}
)
w_c6_run = widgets.Button(
    description='▶  Generate Chart',
    button_style='success', layout=widgets.Layout(width='170px',height='36px')
)
out_c6 = widgets.Output()

def make_c6(_):
    with out_c6:
        clear_output(wait=True)
        dmax = w_c6_max.value
        disks = list(range(1, dmax+1))
        actual  = [count_hanoi(d) for d in disks]
        theory  = [2**d - 1 for d in disks]

        fig, ax = plt.subplots(figsize=(11,5))
        ax.bar(disks, actual, color='#9b59b6', alpha=0.85, label='Actual moves')
        ax.plot(disks, theory, 'r--o', lw=1.8, markersize=4, label='2ⁿ – 1  (formula)')
        ax.set_xlabel('Number of Disks', fontsize=11)
        ax.set_ylabel('Total Moves', fontsize=11)
        ax.set_title('Tower of Hanoi — Move Count Grows as 2ⁿ – 1', fontweight='bold', fontsize=12)
        ax.legend(fontsize=10); ax.spines[['top','right']].set_visible(False)
        plt.tight_layout(); plt.show()

        assert actual == theory, 'Move counts do not match formula!'
        print(f'Verified: all move counts match 2^n - 1 formula. ✓')
        print(f'At n=20: {2**20 - 1:,} moves needed.')
        print(f'At n=64 (classic problem): {2**64 - 1:,} moves — takes longer than the age of the universe.')

w_c6_run.on_click(make_c6)
display(widgets.VBox([w_c6_max, w_c6_run, out_c6]))
